# Stage 1 — Data verification and partition

**Thesis:** Explainable AI for Early Diabetes Prediction: A Comparative Study of Model Interpretability and Clinical Performance

**Implements:** Chapter 3 §3.2.2, §3.2.4, §3.5.2, §3.7.1 (bullets 1–2), §3.7.2 and decisions D2, D4, D14, P3, S1–S5 (decisions_log.md).

**What this notebook does**
1. Verifies the raw CSV (schema, coding, ranges, missingness, repeated rows, class balance). The raw file is only read, never modified.
2. Creates the stratified 80:20 development–test split, the 5 development folds, the 50,000-row tuning subsample (KNN, MLP), the repeated-profile flags for the test rows, and the SHAP index sets.
3. Saves every index set as CSV. **Later stages load these files; they never recompute them.**
4. Logs the environment and pins library versions (`requirements.txt`).
5. Writes the Stage 1 data report and the results-registry rows.

**Run on Google Colab:** upload this notebook, put the dataset CSV in `MyDrive/thesis_xai/data/`, and *Run all*. If `MyDrive/thesis_xai/stage1_outputs/indices/` already holds the canonical index files, the notebook recomputes them, checks they are byte-identical, and never overwrites them.

**Run on the PC:** first `pip install -r requirements.txt` (the Colab-generated file), then run this notebook from the project folder (the folder that contains `data/` and `stage1_outputs/`).

`row_id` in every index file = 0-based data-row number in the raw CSV (header excluded). Later stages select rows with `df.iloc[row_id]`.

In [1]:
# ---------- 0. Environment and paths ----------
import os, sys, subprocess, importlib.util, platform, json, hashlib, datetime as dt

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_DIR = "/content/drive/MyDrive/thesis_xai"
    MACHINE = "colab"
else:
    PROJECT_DIR = os.environ.get("THESIS_PROJECT_DIR", os.path.abspath("."))
    MACHINE = os.environ.get("THESIS_MACHINE", "pc")
os.makedirs(PROJECT_DIR, exist_ok=True)

# Libraries used by later stages. On Colab, missing ones are installed now and then pinned (O2).
# On the PC they must come from requirements.txt, so the notebook stops instead of installing the latest versions.
LATER_STAGE_LIBS = ["xgboost", "lightgbm", "catboost", "shap", "lime", "statsmodels"]
missing = [m for m in LATER_STAGE_LIBS if importlib.util.find_spec(m) is None]
if missing and IN_COLAB:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
elif missing:
    raise SystemExit(f"Missing libraries {missing}. Run: pip install -r requirements.txt")

print("Machine:", MACHINE, "| Project dir:", PROJECT_DIR, "| Python", platform.python_version())

Machine: pc | Project dir: C:\Users\USER\Downloads\thesis_xai | Python 3.13.0


In [2]:
# ---------- 1. Configuration (frozen; version it if anything changes) ----------
import numpy as np
import pandas as pd

CONFIG_VERSION = "v1.0"
MASTER_SEED = 42                     # S2
TEST_SIZE = 0.20                     # Ch3 3.2.4
N_FOLDS = 5                          # Ch3 3.2.4
SUBSAMPLE_N = 50_000                 # D14 (KNN, MLP tuning)
SHAP_BG_N, PERM_BG_N = 500, 100      # D4, P3
GLOBAL_N, PERM_CORE_N = 2_000, 500   # D4
N_STABILITY = 5                      # D4, P3
EXPECTED_SHA256 = "19f367e3e3350768f0c144c5d73ee5b355f67a57eaaa86ca7bd8aec594d8b1d0"          # SHA-256 of the raw CSV used for the canonical run

TARGET = "Diabetes_binary"
EXPECTED_COLUMNS = ["Diabetes_binary", "HighBP", "HighChol", "CholCheck", "BMI", "Smoker", "Stroke",
                    "HeartDiseaseorAttack", "PhysActivity", "Fruits", "Veggies", "HvyAlcoholConsump",
                    "AnyHealthcare", "NoDocbcCost", "GenHlth", "MentHlth", "PhysHlth", "DiffWalk", "Sex",
                    "Age", "Education", "Income"]
FEATURES = [c for c in EXPECTED_COLUMNS if c != TARGET]
BINARY = ["HighBP", "HighChol", "CholCheck", "Smoker", "Stroke", "HeartDiseaseorAttack", "PhysActivity",
          "Fruits", "Veggies", "HvyAlcoholConsump", "AnyHealthcare", "NoDocbcCost", "DiffWalk", "Sex"]
ORDINAL = {"GenHlth": (1, 5), "Age": (1, 13), "Education": (1, 6), "Income": (1, 8)}
NUMERIC = {"BMI": (12, 98), "MentHlth": (0, 30), "PhysHlth": (0, 30)}

# Facts stated in the project brief (checked from the file before Stage 1); verified again below.
EXPECTED_FACTS = {"rows": 253_680, "positives": 35_346, "negatives": 218_334, "repeated_rows": 24_206}

# Stage-specific seeds from one master seed (Ch3 3.7.2). The list is append-only:
# SeedSequence.generate_state is prefix-stable, so adding a stream never changes existing seeds.
SEED_STREAMS = ["partition", "folds", "subsample_50k", "shap_background", "shap_perm_background",
                "global_sample", "perm_core", "stability_bootstrap", "search_sampling",
                "estimator_init", "eval_bootstrap", "lime_case_selection", "lime_perturbation"]
SEEDS = {k: int(v) for k, v in zip(SEED_STREAMS,
                                    np.random.SeedSequence(MASTER_SEED).generate_state(len(SEED_STREAMS)))}

OUT_DIR = os.path.join(PROJECT_DIR, "stage1_outputs")
IDX_DIR, TAB_DIR, ENV_DIR = (os.path.join(OUT_DIR, d) for d in ("indices", "tables", "env"))
for d in (IDX_DIR, TAB_DIR, ENV_DIR):
    os.makedirs(d, exist_ok=True)

CONFIG = dict(config_version=CONFIG_VERSION, master_seed=MASTER_SEED, seeds=SEEDS, test_size=TEST_SIZE,
              n_folds=N_FOLDS, cv="StratifiedKFold(shuffle=True)", subsample_n=SUBSAMPLE_N,
              subsample_strata="target x fold", shap_background_n=SHAP_BG_N, perm_background_n=PERM_BG_N,
              global_sample_n=GLOBAL_N, perm_core_n=PERM_CORE_N, n_stability_resamples=N_STABILITY,
              sampling="proportional stratification by target", repeated_profile_definition=
              "primary: identical values on all 21 predictors; secondary: identical on all 22 columns",
              row_id="0-based data-row number in the raw CSV")
cfg_path = os.path.join(OUT_DIR, f"config_stage1_{CONFIG_VERSION}.json")
if not os.path.exists(cfg_path):
    with open(cfg_path, "w") as f:
        json.dump(CONFIG, f, indent=2)
pd.Series(SEEDS, name="seed").rename_axis("stream").to_frame()

,seed
stream,
partition,3444837047
folds,2669555309
subsample_50k,2046530742
shap_background,3581440988
shap_perm_background,1691623607
global_sample,2099784219
perm_core,1184028159
stability_bootstrap,862288241
search_sampling,1333843299


In [3]:
# ---------- 2. Load the raw CSV (read-only) and checksum it ----------
CANDIDATE_NAMES = ["diabetes_binary_health_indicators_BRFSS2015.csv",
                   "diabetes_binary_health_indicators_BRFSS2015 1.csv",
                   "diabetes_binary_health_indicators_BRFSS2015_1.csv"]
search_dirs = [os.path.join(PROJECT_DIR, "data"), PROJECT_DIR, "/content", "/mnt/user-data/uploads"]
DATA_PATH = os.environ.get("THESIS_DATA_PATH") or next(
    (os.path.join(d, n) for d in search_dirs for n in CANDIDATE_NAMES if os.path.exists(os.path.join(d, n))), None)
assert DATA_PATH, f"Dataset not found. Put it in {os.path.join(PROJECT_DIR, 'data')}"

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

DATA_SHA256 = sha256_file(DATA_PATH)
df = pd.read_csv(DATA_PATH)
print(DATA_PATH, "\nSHA-256:", DATA_SHA256, "\nShape:", df.shape)
if EXPECTED_SHA256 not in ("", "19f367e3e3350768f0c144c5d73ee5b355f67a57eaaa86ca7bd8aec594d8b1d0") and DATA_SHA256 != EXPECTED_SHA256:
    raise SystemExit("This CSV differs from the file used for the canonical Stage 1 run. Stop and check the file.")

C:\Users\USER\Downloads\thesis_xai\data\diabetes_binary_health_indicators_BRFSS2015.csv 
SHA-256: 19f367e3e3350768f0c144c5d73ee5b355f67a57eaaa86ca7bd8aec594d8b1d0 
Shape: (253680, 22)


In [4]:
# ---------- 3. Verification checks (deterministic; nothing is learned from the data) ----------
checks = []
def check(name, expected, observed, ok):
    checks.append({"check": name, "expected": str(expected), "observed": str(observed), "result": "PASS" if ok else "FAIL"})

n_rows = len(df)
y_all = df[TARGET]
check("Rows", f"{EXPECTED_FACTS['rows']:,}", f"{n_rows:,}", n_rows == EXPECTED_FACTS["rows"])
check("Columns (names and order)", "22 (target + 21 predictors)", len(df.columns), list(df.columns) == EXPECTED_COLUMNS)
check("Missing values", 0, int(df.isna().sum().sum()), df.isna().sum().sum() == 0)
dtypes = sorted({str(t) for t in df.dtypes})
check("Storage type", "float64 for all columns", ", ".join(dtypes), dtypes == ["float64"])
is_int = bool(np.all(np.isfinite(df.to_numpy())) and np.all(df.to_numpy() == np.round(df.to_numpy())))
check("All values finite and integer-valued", True, is_int, is_int)
tv = sorted(y_all.unique().tolist())
check("Target coding", "{0, 1}", "{" + ", ".join(f"{v:g}" for v in tv) + "}", tv == [0.0, 1.0])
for c in BINARY:
    vals = sorted(df[c].unique().tolist())
    check(f"{c} values", "{0, 1}", "{" + ", ".join(f"{v:g}" for v in vals) + "}", vals == [0.0, 1.0])
for c, (lo, hi) in {**ORDINAL, **NUMERIC}.items():
    mn, mx = df[c].min(), df[c].max()
    exp = f"{lo}–{hi}"
    check(f"{c} range", exp, f"{mn:g}–{mx:g}", (mn >= lo) and (mx <= hi))
for c, (lo, hi) in ORDINAL.items():
    lv = sorted(df[c].unique().astype(int).tolist())
    check(f"{c} levels present", f"all of {lo}–{hi}", f"{len(lv)} levels", lv == list(range(lo, hi + 1)))
n_pos, n_neg = int((y_all == 1).sum()), int((y_all == 0).sum())
check("Positive class (1)", f"{EXPECTED_FACTS['positives']:,}", f"{n_pos:,} ({100*n_pos/n_rows:.2f}%)", n_pos == EXPECTED_FACTS["positives"])
check("Negative class (0)", f"{EXPECTED_FACTS['negatives']:,}", f"{n_neg:,} ({100*n_neg/n_rows:.2f}%)", n_neg == EXPECTED_FACTS["negatives"])
n_rep_full = int(df.duplicated().sum())
check("Exact repeated rows (all 22 columns, copies beyond the first)", f"{EXPECTED_FACTS['repeated_rows']:,}", f"{n_rep_full:,}", n_rep_full == EXPECTED_FACTS["repeated_rows"])
check("Predictor counts (binary / ordinal / numerical)", "14 / 4 / 3", f"{len(BINARY)} / {len(ORDINAL)} / {len(NUMERIC)}",
      (len(BINARY), len(ORDINAL), len(NUMERIC)) == (14, 4, 3) and set(BINARY) | set(ORDINAL) | set(NUMERIC) == set(FEATURES))

checks_df = pd.DataFrame(checks)
checks_df.to_csv(os.path.join(TAB_DIR, "s1_verification_checks.csv"), index=False)
n_fail = int((checks_df.result == "FAIL").sum())
print(f"{len(checks_df)} checks, {n_fail} failed")
assert n_fail == 0, checks_df[checks_df.result == "FAIL"]
checks_df

35 checks, 0 failed


,check,expected,observed,result
0,Rows,"253,680","253,680",PASS
1,Columns (names and order),22 (target + 21 predictors),22,PASS
2,Missing values,0,0,PASS
3,Storage type,float64 for all columns,float64,PASS
4,All values finite and integer-valued,True,True,PASS
5,Target coding,"{0, 1}","{0, 1}",PASS
6,HighBP values,"{0, 1}","{0, 1}",PASS
7,HighChol values,"{0, 1}","{0, 1}",PASS
8,CholCheck values,"{0, 1}","{0, 1}",PASS
9,Smoker values,"{0, 1}","{0, 1}",PASS


In [5]:
# ---------- 4. Repeated rows, BMI and day-count distributions (descriptive) ----------
y = y_all.astype(int).to_numpy()
pid_feat = df.groupby(FEATURES, sort=False).ngroup().to_numpy()          # identical 21-predictor profile
pid_row = df.groupby(EXPECTED_COLUMNS, sort=False).ngroup().to_numpy()   # identical full row

rep = {}
rep["full_repeated_rows"] = n_rep_full
rep["full_rows_in_repeated_groups"] = int(df.duplicated(keep=False).sum())
rep["full_distinct_rows"] = int(len(np.unique(pid_row)))
rep["full_repeated_groups"] = int((pd.Series(pid_row).value_counts() > 1).sum())
rep["full_largest_group"] = int(pd.Series(pid_row).value_counts().max())
dup_mask = df.duplicated().to_numpy()
rep["full_repeated_rows_positive"] = int(y[dup_mask].sum())
rep["feat_repeated_rows"] = int(df[FEATURES].duplicated().sum())
rep["feat_distinct_profiles"] = int(len(np.unique(pid_feat)))
lab_per_profile = pd.Series(y).groupby(pid_feat).nunique()
conflict_ids = lab_per_profile.index[lab_per_profile > 1]
rep["feat_conflicting_profiles"] = int(len(conflict_ids))
rep["feat_rows_in_conflicting_profiles"] = int(np.isin(pid_feat, conflict_ids).sum())
rep_df = pd.Series(rep, name="value").rename_axis("quantity").reset_index()
rep_df.to_csv(os.path.join(TAB_DIR, "s1_repeated_rows.csv"), index=False)

pct = [1, 5, 25, 50, 75, 95, 99]
bmi = df["BMI"]
bmi_stats = {f"p{p}": float(np.percentile(bmi, p)) for p in pct}
bmi_stats.update(min=float(bmi.min()), max=float(bmi.max()), mean=float(bmi.mean()), sd=float(bmi.std()),
                 n_ge_50=int((bmi >= 50).sum()), n_ge_70=int((bmi >= 70).sum()), n_ge_90=int((bmi >= 90).sum()))
days = {f"{c}_{k}": v for c in ("MentHlth", "PhysHlth")
        for k, v in (("pct_zero", 100 * float((df[c] == 0).mean())), ("pct_30", 100 * float((df[c] == 30).mean())))}
dist_df = pd.Series({**{f"BMI_{k}": v for k, v in bmi_stats.items()}, **days}, name="value").rename_axis("quantity").reset_index()
dist_df.to_csv(os.path.join(TAB_DIR, "s1_bmi_days_distribution.csv"), index=False)
from IPython.display import display
display(rep_df)
dist_df

,quantity,value
0,full_repeated_rows,24206
1,full_rows_in_repeated_groups,35575
2,full_distinct_rows,229474
3,full_repeated_groups,11369
4,full_largest_group,59
5,full_repeated_rows_positive,249
6,feat_repeated_rows,25772
7,feat_distinct_profiles,227908
8,feat_conflicting_profiles,1566
9,feat_rows_in_conflicting_profiles,5218


,quantity,value
0,BMI_p1,18.000000
1,BMI_p5,20.000000
2,BMI_p25,24.000000
3,BMI_p50,27.000000
4,BMI_p75,31.000000
5,BMI_p95,40.000000
6,BMI_p99,50.000000
7,BMI_min,12.000000
8,BMI_max,98.000000
9,BMI_mean,28.382364


In [6]:
# ---------- 5. Descriptive statistics by class (whole file; descriptive only, never used for modelling) ----------
rows = []
for c in FEATURES:
    kind = "binary" if c in BINARY else ("ordinal" if c in ORDINAL else "numerical")
    r = {"variable": c, "type": kind}
    for lab, sub in (("all", df), ("class0", df[y == 0]), ("class1", df[y == 1])):
        if kind == "binary":
            r[f"{lab}_pct_yes"] = 100 * sub[c].mean()
        else:
            q1, md, q3 = np.percentile(sub[c], [25, 50, 75])
            r.update({f"{lab}_mean": sub[c].mean(), f"{lab}_sd": sub[c].std(), f"{lab}_median": md,
                      f"{lab}_q1": q1, f"{lab}_q3": q3})
    rows.append(r)
desc_df = pd.DataFrame(rows)
desc_df.round(3).to_csv(os.path.join(TAB_DIR, "s1_descriptives_by_class.csv"), index=False)
desc_df.round(2)

,variable,type,all_pct_yes,class0_pct_yes,class1_pct_yes,all_mean,all_sd,all_median,all_q1,all_q3,class0_mean,class0_sd,class0_median,class0_q1,class0_q3,class1_mean,class1_sd,class1_median,class1_q1,class1_q3
0,HighBP,binary,42.90,37.66,75.27,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,HighChol,binary,42.41,38.43,67.01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,CholCheck,binary,96.27,95.77,99.32,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,BMI,numerical,NaN,NaN,NaN,28.38,6.61,27.0,24.0,31.0,27.81,6.29,27.0,24.0,31.0,31.94,7.36,31.0,27.0,35.0
4,Smoker,binary,44.32,43.10,51.82,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,Stroke,binary,4.06,3.22,9.25,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,HeartDiseaseorAttack,binary,9.42,7.34,22.29,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,PhysActivity,binary,75.65,77.69,63.05,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,Fruits,binary,63.43,64.22,58.54,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,Veggies,binary,81.14,82.03,75.64,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [7]:
# ---------- 6. Partition: 80:20 stratified split, 5 folds, 50k subsample ----------
from sklearn.model_selection import train_test_split, StratifiedKFold

row_id = np.arange(n_rows)
dev_ids, test_ids = train_test_split(row_id, test_size=TEST_SIZE, stratify=y, shuffle=True,
                                     random_state=SEEDS["partition"])
dev_ids, test_ids = np.sort(dev_ids), np.sort(test_ids)
assert len(np.intersect1d(dev_ids, test_ids)) == 0 and len(dev_ids) + len(test_ids) == n_rows

split_df = pd.DataFrame({"row_id": row_id, "partition": np.where(np.isin(row_id, test_ids), "test", "development"), "y": y})

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEEDS["folds"])
fold = np.full(len(dev_ids), -1)
for k, (_, va) in enumerate(skf.split(dev_ids, y[dev_ids])):
    fold[va] = k
assert (fold >= 0).all()
folds_df = pd.DataFrame({"row_id": dev_ids, "fold": fold, "y": y[dev_ids]})

# 50k subsample: stratified by target x fold, keeps each row's development fold (D14, run plan §6)
pos_in_dev, _ = train_test_split(np.arange(len(dev_ids)), train_size=SUBSAMPLE_N,
                                 stratify=y[dev_ids] * N_FOLDS + fold, random_state=SEEDS["subsample_50k"])
sub_df = folds_df.iloc[np.sort(pos_in_dev)].reset_index(drop=True)

In [8]:
# ---------- 7. Repeated-profile flags for test rows (D2 sensitivity analysis) ----------
dev_feat = pd.Series(pid_feat[dev_ids]).value_counts()
dev_row_set = set(pid_row[dev_ids].tolist())
flags_df = pd.DataFrame({
    "row_id": test_ids,
    "y": y[test_ids],
    "feature_profile_in_dev": np.isin(pid_feat[test_ids], dev_feat.index).astype(int),   # primary (S3)
    "full_row_in_dev": np.array([p in dev_row_set for p in pid_row[test_ids]], dtype=int), # secondary
    "n_dev_rows_same_profile": pd.Series(pid_feat[test_ids]).map(dev_feat).fillna(0).astype(int).to_numpy(),
})

def overlap_summary(f):
    out = {}
    for lab, m in (("all", np.ones(len(f), bool)), ("class0", f.y == 0), ("class1", f.y == 1)):
        g = f[m]
        out[lab] = {"n_test": len(g),
                    "feature_profile_in_dev": int(g.feature_profile_in_dev.sum()),
                    "full_row_in_dev": int(g.full_row_in_dev.sum()),
                    "novel_feature_profile": int((g.feature_profile_in_dev == 0).sum())}
    return pd.DataFrame(out).T
overlap_df = overlap_summary(flags_df)
overlap_df["pct_feature_profile_in_dev"] = 100 * overlap_df.feature_profile_in_dev / overlap_df.n_test
overlap_df["pct_full_row_in_dev"] = 100 * overlap_df.full_row_in_dev / overlap_df.n_test
overlap_df.round(3).to_csv(os.path.join(TAB_DIR, "s1_test_profile_overlap.csv"))
overlap_df.round(2)

,n_test,feature_profile_in_dev,full_row_in_dev,novel_feature_profile,pct_feature_profile_in_dev,pct_full_row_in_dev
all,50736,6919,6470,43817,13.64,12.75
class0,43667,6594,6402,37073,15.10,14.66
class1,7069,325,68,6744,4.60,0.96


In [9]:
# ---------- 8. SHAP index sets (D4, P3), frozen before any model exists ----------
def strat_sample(ids, n, seed):
    sel, _ = train_test_split(ids, train_size=n, stratify=y[ids], random_state=seed)
    return np.sort(sel)

bg_ids = strat_sample(dev_ids, SHAP_BG_N, SEEDS["shap_background"])
perm_bg_ids = strat_sample(bg_ids, PERM_BG_N, SEEDS["shap_perm_background"])
glob_ids = strat_sample(test_ids, GLOBAL_N, SEEDS["global_sample"])
core_ids = strat_sample(glob_ids, PERM_CORE_N, SEEDS["perm_core"])
assert set(perm_bg_ids) <= set(bg_ids) and set(core_ids) <= set(glob_ids)
assert set(bg_ids) <= set(dev_ids) and set(glob_ids) <= set(test_ids)

def as_df(ids):
    return pd.DataFrame({"row_id": ids, "y": y[ids]})

def stratified_bootstraps(ids, set_code):
    """N_STABILITY bootstrap resamples drawn with replacement within each class (class counts kept)."""
    out = []
    for b in range(N_STABILITY):
        rng = np.random.default_rng([SEEDS["stability_bootstrap"], set_code, b])
        draw = np.concatenate([rng.choice(ids[y[ids] == c], size=int((y[ids] == c).sum()), replace=True) for c in (0, 1)])
        out.append(pd.DataFrame({"resample": b, "position": np.arange(len(draw)), "row_id": draw, "y": y[draw]}))
    return pd.concat(out, ignore_index=True)

boot_glob_df = stratified_bootstraps(glob_ids, 1)
boot_core_df = stratified_bootstraps(core_ids, 2)

In [10]:
# ---------- 9. Save index files (create, or verify against existing canonical files; never overwrite) ----------
INDEX_FILES = {
    "split_dev_test.csv": split_df,
    "dev_folds.csv": folds_df,
    "dev_subsample_50k.csv": sub_df,
    "test_repeated_profile_flags.csv": flags_df,
    "shap_background_500_dev.csv": as_df(bg_ids),
    "shap_perm_background_100_dev.csv": as_df(perm_bg_ids),
    "shap_global_sample_2000_test.csv": as_df(glob_ids),
    "shap_perm_core_500_test.csv": as_df(core_ids),
    "shap_stability_bootstrap_global_2000.csv": boot_glob_df,
    "shap_stability_bootstrap_core_500.csv": boot_core_df,
}
manifest = []
for name, frame in INDEX_FILES.items():
    data = frame.to_csv(index=False, lineterminator="\n").encode()
    h = hashlib.sha256(data).hexdigest()
    path = os.path.join(IDX_DIR, name)
    if os.path.exists(path):
        old = hashlib.sha256(open(path, "rb").read()).hexdigest()
        status = "verified: identical to existing file" if old == h else "MISMATCH: existing file kept"
    else:
        with open(path, "wb") as f:
            f.write(data)
        status = "created"
    manifest.append({"file": name, "rows": len(frame), "sha256": h, "status": status})
manifest_df = pd.DataFrame(manifest)
manifest_path = os.path.join(IDX_DIR, "MANIFEST.csv")
if not os.path.exists(manifest_path):
    manifest_df[["file", "rows", "sha256"]].to_csv(manifest_path, index=False, lineterminator="\n")
manifest_df.to_csv(os.path.join(ENV_DIR, f"index_check_{MACHINE}.csv"), index=False)
if (manifest_df.status.str.startswith("MISMATCH")).any():
    raise SystemExit("Recomputed indices differ from the canonical files. Keep the canonical files; compare library versions.")
manifest_df

,file,rows,sha256,status
0,split_dev_test.csv,253680,f73c8f475217b3ccada3baa34ac26e615614a8c400aea3...,verified: identical to existing file
1,dev_folds.csv,202944,c71320238e98f1d8fedf1ce72475f8572e275248052014...,verified: identical to existing file
2,dev_subsample_50k.csv,50000,982d9518ce9d838302e49f78080b60882c5e79845d765a...,verified: identical to existing file
3,test_repeated_profile_flags.csv,50736,2aa16f96ba8b824efb7df8cacc2f2503d72080cf01b748...,verified: identical to existing file
4,shap_background_500_dev.csv,500,981fe21fcd3e157857be057b0a9e43a93115fe99e10aff...,verified: identical to existing file
5,shap_perm_background_100_dev.csv,100,a5c697d56d71fa26074110fef9e18f16cdfd3fbdff6c11...,verified: identical to existing file
6,shap_global_sample_2000_test.csv,2000,a019718a439d8c48519f538db254b5c63d560f880f9193...,verified: identical to existing file
7,shap_perm_core_500_test.csv,500,59dae554426fe5adbbc9330505ccfaa099b9022d9e8e76...,verified: identical to existing file
8,shap_stability_bootstrap_global_2000.csv,10000,a2a2d29b66dfe2ebf2cee100e28e5f1b082d00174b2b68...,verified: identical to existing file
9,shap_stability_bootstrap_core_500.csv,2500,cd5ede4c710969073e25656de89ef67063fbf8d714f6c6...,verified: identical to existing file


In [11]:
# ---------- 10. Balance of every index set ----------
def bal(name, source, ids):
    yy = y[np.asarray(ids)]
    return {"set": name, "drawn_from": source, "n": len(yy), "n_pos": int(yy.sum()), "n_neg": int((yy == 0).sum()),
            "pct_pos": 100 * yy.mean()}
balance = [bal("Full file", "-", row_id), bal("Development", "full file", dev_ids), bal("Test", "full file", test_ids)]
balance += [bal(f"Development fold {k}", "development", dev_ids[fold == k]) for k in range(N_FOLDS)]
balance += [bal("Tuning subsample (KNN, MLP)", "development", sub_df.row_id)]
balance += [bal(f"Subsample fold {k}", "subsample", sub_df.row_id[sub_df.fold == k]) for k in range(N_FOLDS)]
balance += [bal("SHAP background", "development", bg_ids), bal("Permutation background", "SHAP background", perm_bg_ids),
            bal("Global explanation sample", "test", glob_ids), bal("Permutation core sample", "global sample", core_ids),
            bal("Test rows, novel 21-predictor profile", "test", flags_df.row_id[flags_df.feature_profile_in_dev == 0])]
balance_df = pd.DataFrame(balance)
balance_df.round(3).to_csv(os.path.join(TAB_DIR, "s1_partition_balance.csv"), index=False)
balance_df.round(2)

,set,drawn_from,n,n_pos,n_neg,pct_pos
0,Full file,-,253680,35346,218334,13.93
1,Development,full file,202944,28277,174667,13.93
2,Test,full file,50736,7069,43667,13.93
3,Development fold 0,development,40589,5655,34934,13.93
4,Development fold 1,development,40589,5655,34934,13.93
5,Development fold 2,development,40589,5656,34933,13.93
6,Development fold 3,development,40589,5656,34933,13.93
7,Development fold 4,development,40588,5655,34933,13.93
8,"Tuning subsample (KNN, MLP)",development,50000,6965,43035,13.93
9,Subsample fold 0,subsample,10000,1393,8607,13.93


In [12]:
# ---------- 11. Environment log and requirements.txt (S1b) ----------
from importlib import metadata
PIN = ["numpy", "pandas", "scipy", "scikit-learn", "xgboost", "lightgbm", "catboost", "shap", "lime",
       "statsmodels", "matplotlib", "joblib", "threadpoolctl", "numba", "llvmlite", "cloudpickle", "tqdm"]
versions = {}
for p in PIN:
    try:
        versions[p] = metadata.version(p)
    except metadata.PackageNotFoundError:
        versions[p] = None

def ram_gb():
    try:
        return round(os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES") / 1e9, 1)
    except (ValueError, OSError, AttributeError):
        return None
try:
    gpu = subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"], capture_output=True, text=True).stdout.strip() or None
except FileNotFoundError:
    gpu = None
now_utc = dt.datetime.now(dt.timezone.utc)
RUN_DATE = now_utc.date().isoformat()
env = {"stage": 1, "machine": MACHINE, "run_utc": now_utc.isoformat(timespec="seconds"),
       "python": platform.python_version(), "platform": platform.platform(), "processor": platform.processor() or platform.machine(),
       "cpu_count": os.cpu_count(), "ram_gb": ram_gb(), "gpu": gpu, "in_colab": IN_COLAB,
       "dataset_path": DATA_PATH, "dataset_sha256": DATA_SHA256, "config_version": CONFIG_VERSION,
       "master_seed": MASTER_SEED, "packages": versions,
       "index_files": {m["file"]: m["status"] for m in manifest}}
with open(os.path.join(ENV_DIR, f"env_log_stage1_{MACHINE}.json"), "w") as f:
    json.dump(env, f, indent=2)

header = (f"# Stage 1 pins. machine={MACHINE}; python={platform.python_version()}; generated {env['run_utc']}\n"
          + ("# CANONICAL: generated on Google Colab at run time (S1b). Install on the PC with: pip install -r requirements.txt\n"
             if MACHINE == "colab" else
             "# PROVISIONAL: not generated on Colab. Replace with the file written by the Colab run (S1b).\n"))
req_text = header + "".join(f"{p}=={v}\n" for p, v in versions.items() if v)
req_path = os.path.join(PROJECT_DIR, "requirements.txt")
existing_is_colab = os.path.exists(req_path) and "machine=colab" in open(req_path).readline()
if MACHINE == "colab" or not os.path.exists(req_path):
    with open(req_path, "w") as f:
        f.write(req_text)
    print("Wrote", req_path)
else:
    alt = os.path.join(ENV_DIR, f"requirements_{MACHINE}.txt")
    with open(alt, "w") as f:
        f.write(req_text)
    ref = dict(l.strip().split("==") for l in open(req_path) if "==" in l)
    diff = {p: (ref.get(p), v) for p, v in versions.items() if v and ref.get(p) != v}
    print("Wrote", alt, "| differences from requirements.txt (reference, this machine):", diff or "none")
env

Wrote C:\Users\USER\Downloads\thesis_xai\stage1_outputs\env\requirements_pc.txt | differences from requirements.txt (reference, this machine): none


{'stage': 1,
 'machine': 'pc',
 'run_utc': '2026-10-12T14:33:17+00:00',
 'python': '3.13.0',
 'platform': 'Windows-11-10.0.26200-SP0',
 'processor': 'AMD64 Family 23 Model 104 Stepping 1, AuthenticAMD',
 'cpu_count': 12,
 'ram_gb': None,
 'gpu': None,
 'in_colab': False,
 'dataset_path': 'C:\\Users\\USER\\Downloads\\thesis_xai\\data\\diabetes_binary_health_indicators_BRFSS2015.csv',
 'dataset_sha256': '19f367e3e3350768f0c144c5d73ee5b355f67a57eaaa86ca7bd8aec594d8b1d0',
 'config_version': 'v1.0',
 'master_seed': 42,
 'packages': {'numpy': '2.1.3',
  'pandas': '2.2.3',
  'scipy': '1.16.3',
  'scikit-learn': '1.6.1',
  'xgboost': '3.4.1',
  'lightgbm': '4.6.0',
  'catboost': '1.2.10',
  'shap': '0.52.0',
  'lime': '0.2.0.1',
  'statsmodels': '0.15.0',
  'matplotlib': '3.10.0',
  'joblib': '1.6.0',
  'threadpoolctl': '3.7.0',
  'numba': '0.61.2',
  'llvmlite': '0.44.0',
  'cloudpickle': '3.1.2',
  'tqdm': '4.67.3'},
 'index_files': {'split_dev_test.csv': 'verified: identical to existing fil

In [13]:
# ---------- 12. Data report (Markdown) and results-registry rows ----------
SCRIPT = "stage1_data_partition.ipynb"
reg = []
def R(rid, section, desc, value, unit, out, notes=""):
    reg.append({"result_id": rid, "chapter_section": section, "description": desc, "value": value, "unit": unit,
                "script": SCRIPT, "output_file": out, "config_version": CONFIG_VERSION, "run_date": RUN_DATE,
                "machine": MACHINE, "notes": notes})
T = "stage1_outputs/tables/"
R("S1-001", "Ch3 3.2.1; Ch4 data", "Rows in raw CSV", n_rows, "rows", T + "s1_verification_checks.csv")
R("S1-002", "Ch3 3.2.1", "Predictors", len(FEATURES), "count", T + "s1_verification_checks.csv", "14 binary, 4 ordinal, 3 numerical")
R("S1-003", "Ch3 3.2.2", "Missing values", int(df.isna().sum().sum()), "cells", T + "s1_verification_checks.csv")
R("S1-004", "Ch3 3.2.1", "Positive class (prediabetes or diabetes)", n_pos, "rows", T + "s1_verification_checks.csv")
R("S1-005", "Ch3 3.2.1", "Positive class share", round(100 * n_pos / n_rows, 2), "%", T + "s1_verification_checks.csv")
R("S1-006", "Ch3 3.2.1", "Negative class", n_neg, "rows", T + "s1_verification_checks.csv")
R("S1-007", "Ch3 3.2.1", "Negative class share", round(100 * n_neg / n_rows, 2), "%", T + "s1_verification_checks.csv")
R("S1-008", "Ch3 3.2.2", "Exact repeated rows (22 columns, copies beyond first)", rep["full_repeated_rows"], "rows", T + "s1_repeated_rows.csv")
R("S1-009", "Ch3 3.2.2", "Repeated rows as share of file", round(100 * rep["full_repeated_rows"] / n_rows, 2), "%", T + "s1_repeated_rows.csv")
R("S1-010", "Ch3 3.2.2", "Distinct full rows", rep["full_distinct_rows"], "rows", T + "s1_repeated_rows.csv")
R("S1-011", "Ch3 3.2.2", "Rows belonging to a repeated full-row group", rep["full_rows_in_repeated_groups"], "rows", T + "s1_repeated_rows.csv")
R("S1-012", "Ch3 3.2.2", "Repeated rows that are positive", rep["full_repeated_rows_positive"], "rows", T + "s1_repeated_rows.csv")
R("S1-013", "Ch3 3.2.2", "Repeated 21-predictor profiles (copies beyond first)", rep["feat_repeated_rows"], "rows", T + "s1_repeated_rows.csv")
R("S1-014", "Ch3 3.2.2", "Distinct 21-predictor profiles", rep["feat_distinct_profiles"], "profiles", T + "s1_repeated_rows.csv")
R("S1-015", "Ch3 3.2.2; Discussion", "Predictor profiles with both class labels", rep["feat_conflicting_profiles"], "profiles", T + "s1_repeated_rows.csv")
R("S1-016", "Ch3 3.2.2; Discussion", "Rows in profiles with both class labels", rep["feat_rows_in_conflicting_profiles"], "rows", T + "s1_repeated_rows.csv")
R("S1-017", "Ch3 3.2.2", "BMI minimum", bmi_stats["min"], "kg/m2", T + "s1_bmi_days_distribution.csv")
R("S1-018", "Ch3 3.2.2", "BMI maximum", bmi_stats["max"], "kg/m2", T + "s1_bmi_days_distribution.csv")
R("S1-019", "Ch3 3.2.2", "BMI median", bmi_stats["p50"], "kg/m2", T + "s1_bmi_days_distribution.csv")
R("S1-020", "Ch3 3.2.2", "BMI 99th percentile", bmi_stats["p99"], "kg/m2", T + "s1_bmi_days_distribution.csv")
R("S1-021", "Ch3 3.2.2", "Rows with BMI >= 50", bmi_stats["n_ge_50"], "rows", T + "s1_bmi_days_distribution.csv")
R("S1-022", "Ch3 3.2.2", "Rows with BMI >= 70", bmi_stats["n_ge_70"], "rows", T + "s1_bmi_days_distribution.csv")
R("S1-023", "Ch3 3.2.2", "MentHlth = 0 days", round(days["MentHlth_pct_zero"], 2), "%", T + "s1_bmi_days_distribution.csv")
R("S1-024", "Ch3 3.2.2", "PhysHlth = 0 days", round(days["PhysHlth_pct_zero"], 2), "%", T + "s1_bmi_days_distribution.csv")
bd = balance_df.set_index("set")
R("S1-025", "Ch3 3.2.4; Ch4", "Development partition size", int(bd.loc["Development", "n"]), "rows", "stage1_outputs/indices/split_dev_test.csv")
R("S1-026", "Ch3 3.2.4; Ch4", "Development positives", int(bd.loc["Development", "n_pos"]), "rows", "stage1_outputs/indices/split_dev_test.csv")
R("S1-027", "Ch3 3.2.4; Ch4", "Test partition size", int(bd.loc["Test", "n"]), "rows", "stage1_outputs/indices/split_dev_test.csv")
R("S1-028", "Ch3 3.2.4; Ch4", "Test positives", int(bd.loc["Test", "n_pos"]), "rows", "stage1_outputs/indices/split_dev_test.csv")
R("S1-029", "Ch3 3.2.4", "Development positive share", round(bd.loc["Development", "pct_pos"], 2), "%", "stage1_outputs/indices/split_dev_test.csv")
R("S1-030", "Ch3 3.2.4", "Test positive share", round(bd.loc["Test", "pct_pos"], 2), "%", "stage1_outputs/indices/split_dev_test.csv")
fsz = bd.loc[[f"Development fold {k}" for k in range(N_FOLDS)], "n"]
R("S1-031", "Ch3 3.2.4", "Development fold size range", f"{int(fsz.min())}-{int(fsz.max())}", "rows", "stage1_outputs/indices/dev_folds.csv")
fpp = bd.loc[[f"Development fold {k}" for k in range(N_FOLDS)], "pct_pos"]
R("S1-032", "Ch3 3.2.4", "Development fold positive share range", f"{fpp.min():.2f}-{fpp.max():.2f}", "%", "stage1_outputs/indices/dev_folds.csv")
R("S1-033", "Ch3 3.4.2 (D14)", "Tuning subsample size", int(bd.loc["Tuning subsample (KNN, MLP)", "n"]), "rows", "stage1_outputs/indices/dev_subsample_50k.csv")
R("S1-034", "Ch3 3.4.2 (D14)", "Tuning subsample positives", int(bd.loc["Tuning subsample (KNN, MLP)", "n_pos"]), "rows", "stage1_outputs/indices/dev_subsample_50k.csv")
od = overlap_df
R("S1-035", "Ch3 3.2.2; Ch4 sensitivity (D2)", "Test rows whose 21-predictor profile occurs in development", int(od.loc["all", "feature_profile_in_dev"]), "rows", T + "s1_test_profile_overlap.csv")
R("S1-036", "Ch3 3.2.2; Ch4 sensitivity (D2)", "Share of test rows with profile in development", round(od.loc["all", "pct_feature_profile_in_dev"], 2), "%", T + "s1_test_profile_overlap.csv")
R("S1-037", "Ch4 sensitivity (D2)", "Test rows with novel profile (sensitivity subset)", int(od.loc["all", "novel_feature_profile"]), "rows", T + "s1_test_profile_overlap.csv")
R("S1-038", "Ch4 sensitivity (D2)", "Positives in novel-profile test subset", int(od.loc["class1", "novel_feature_profile"]), "rows", T + "s1_test_profile_overlap.csv")
R("S1-039", "Ch4 sensitivity (D2)", "Share of test positives with profile in development", round(od.loc["class1", "pct_feature_profile_in_dev"], 2), "%", T + "s1_test_profile_overlap.csv")
R("S1-040", "Ch4 sensitivity (D2)", "Share of test negatives with profile in development", round(od.loc["class0", "pct_feature_profile_in_dev"], 2), "%", T + "s1_test_profile_overlap.csv")
R("S1-041", "Ch4 sensitivity (D2)", "Test rows whose full row (incl. label) occurs in development", int(od.loc["all", "full_row_in_dev"]), "rows", T + "s1_test_profile_overlap.csv")
for rid, nm in (("S1-042", "SHAP background"), ("S1-043", "Permutation background"),
                ("S1-044", "Global explanation sample"), ("S1-045", "Permutation core sample")):
    R(rid, "Ch3 3.5.2; Ch4 XAI", f"{nm}: size / positives", f"{int(bd.loc[nm, 'n'])} / {int(bd.loc[nm, 'n_pos'])}", "rows", T + "s1_partition_balance.csv")
R("S1-046", "Ch3 3.7.2", "Master seed", MASTER_SEED, "-", f"stage1_outputs/config_stage1_{CONFIG_VERSION}.json")
R("S1-047", "Ch3 3.7.2", "Dataset SHA-256", DATA_SHA256, "-", f"stage1_outputs/env/env_log_stage1_{MACHINE}.json")
R("S1-048", "Ch3 3.2.2", "BMI 1st percentile", bmi_stats["p1"], "kg/m2", T + "s1_bmi_days_distribution.csv")
R("S1-049", "Ch3 3.2.2", "Rows with BMI >= 90", bmi_stats["n_ge_90"], "rows", T + "s1_bmi_days_distribution.csv")
R("S1-050", "Ch4 data", "BMI mean", round(bmi_stats["mean"], 2), "kg/m2", T + "s1_bmi_days_distribution.csv")
R("S1-051", "Ch4 data", "BMI SD", round(bmi_stats["sd"], 2), "kg/m2", T + "s1_bmi_days_distribution.csv")
R("S1-052", "Ch3 3.2.2", "MentHlth = 30 days", round(days["MentHlth_pct_30"], 2), "%", T + "s1_bmi_days_distribution.csv")
R("S1-053", "Ch3 3.2.2", "PhysHlth = 30 days", round(days["PhysHlth_pct_30"], 2), "%", T + "s1_bmi_days_distribution.csv")
R("S1-054", "Ch3 3.2.5; run plan 6", "Negative/positive ratio (bound for positive-class weight)", round(n_neg / n_pos, 2), "ratio", T + "s1_verification_checks.csv")
R("S1-055", "Ch3 3.2.2", "Repeated full-row groups", rep["full_repeated_groups"], "groups", T + "s1_repeated_rows.csv")
R("S1-056", "Ch3 3.2.2", "Largest repeated full-row group", rep["full_largest_group"], "rows", T + "s1_repeated_rows.csv")
R("S1-057", "Ch4 sensitivity (D2)", "Positive share in novel-profile test subset", round(bd.loc["Test rows, novel 21-predictor profile", "pct_pos"], 2), "%", T + "s1_partition_balance.csv")
R("S1-058", "Ch4 sensitivity (D2)", "Test negatives with profile in development", int(od.loc["class0", "feature_profile_in_dev"]), "rows", T + "s1_test_profile_overlap.csv")
R("S1-059", "Ch4 sensitivity (D2)", "Test positives with profile in development", int(od.loc["class1", "feature_profile_in_dev"]), "rows", T + "s1_test_profile_overlap.csv")
R("S1-060", "Ch3 3.4.2 (D14)", "Tuning subsample fold sizes (each)", int(bd.loc["Subsample fold 0", "n"]), "rows", T + "s1_partition_balance.csv", "all five folds 10,000 rows, 1,393 positive")
R("S1-061", "Ch4 data (Table)", "Descriptive statistics by class, 21 predictors (table; values in file)", "see file", "-", T + "s1_descriptives_by_class.csv", "whole file, descriptive only")
R("S1-062", "Ch3 3.2.4 (Table)", "Per-fold and per-set class counts (table; values in file)", "see file", "-", T + "s1_partition_balance.csv")
R("S1-063", "Ch3 3.2.2 (Table)", "Verification checks passed", f"{len(checks_df) - n_fail}/{len(checks_df)}", "checks", T + "s1_verification_checks.csv")
reg_df = pd.DataFrame(reg)
reg_df.to_csv(os.path.join(OUT_DIR, "stage1_registry_rows.csv"), index=False)
print(len(reg_df), "registry rows")

63 registry rows


In [14]:
# ---------- 12b. Write the Markdown report ----------
def md_table(frame):
    cols = list(frame.columns)
    lines = ["| " + " | ".join(map(str, cols)) + " |", "| " + " | ".join("---" for _ in cols) + " |"]
    lines += ["| " + " | ".join(str(v) for v in r) + " |" for r in frame.itertuples(index=False)]
    return "\n".join(lines)
f0 = lambda v: f"{int(v):,}"
f2 = lambda v: f"{v:.2f}"

CODEBOOK = [  # variable, coding in the file, source (see report §2 sources)
    ("Diabetes_binary", "0 = no diabetes; 1 = prediabetes or diabetes (ever told)", "UCI"),
    ("HighBP / HighChol", "0 = no, 1 = yes (high blood pressure / high cholesterol)", "UCI"),
    ("CholCheck", "1 = cholesterol check within 5 years", "UCI"),
    ("BMI", "Body mass index, kg/m²; CDC _BMI5 has two implied decimals and treats < 12 or ≥ 100 as missing; values in this file are whole numbers", "UCI; CDC-CV"),
    ("Smoker", "1 = smoked ≥ 100 cigarettes in lifetime", "UCI"),
    ("Stroke / HeartDiseaseorAttack", "1 = ever told stroke / CHD or MI", "UCI"),
    ("PhysActivity", "1 = physical activity in past 30 days, excluding job", "UCI"),
    ("Fruits / Veggies", "1 = consumes fruit / vegetables ≥ 1 time per day", "UCI"),
    ("HvyAlcoholConsump", "1 = men > 14, women > 7 drinks per week", "UCI"),
    ("AnyHealthcare / NoDocbcCost", "1 = has any coverage / could not see a doctor because of cost (12 months)", "UCI"),
    ("GenHlth", "1 = excellent, 2 = very good, 3 = good, 4 = fair, 5 = poor", "UCI; CDC-CB (GENHLTH)"),
    ("MentHlth / PhysHlth", "Days not good in past 30 (0–30); CDC code 88 = 'None' appears as 0 here", "UCI; CDC-CB"),
    ("DiffWalk", "1 = serious difficulty walking or climbing stairs", "UCI"),
    ("Sex", "0 = female, 1 = male (CDC SEX: 1 = male, 2 = female)", "UCI; CDC-CB"),
    ("Age", "1 = 18–24, 2 = 25–29, 3 = 30–34, 4 = 35–39, 5 = 40–44, 6 = 45–49, 7 = 50–54, 8 = 55–59, 9 = 60–64, 10 = 65–69, 11 = 70–74, 12 = 75–79, 13 = 80 or older (code 14 'don't know/refused' absent)", "CDC-CV (_AGEG5YR); UCI"),
    ("Education", "1 = never attended / kindergarten only, 2 = grades 1–8, 3 = grades 9–11, 4 = grade 12 or GED, 5 = college 1–3 years, 6 = college 4+ years", "CDC-CB (EDUCA); UCI"),
    ("Income", "1 = < $10k, 2 = $10k–<15k, 3 = $15k–<20k, 4 = $20k–<25k, 5 = $25k–<35k, 6 = $35k–<50k, 7 = $50k–<75k, 8 = ≥ $75k", "CDC-CB (INCOME2); UCI"),
]

desc_md = []
for r in desc_df.itertuples(index=False):
    if r.type == "binary":
        desc_md.append((r.variable, r.type, f"{r.all_pct_yes:.1f}%", f"{r.class0_pct_yes:.1f}%", f"{r.class1_pct_yes:.1f}%"))
    else:
        fmt = lambda m, s, md, a, b: f"{m:.2f} ({s:.2f}); {md:g} [{a:g}–{b:g}]"
        desc_md.append((r.variable, r.type, fmt(r.all_mean, r.all_sd, r.all_median, r.all_q1, r.all_q3),
                        fmt(r.class0_mean, r.class0_sd, r.class0_median, r.class0_q1, r.class0_q3),
                        fmt(r.class1_mean, r.class1_sd, r.class1_median, r.class1_q1, r.class1_q3)))
desc_md = pd.DataFrame(desc_md, columns=["Variable", "Type", "All", "Class 0", "Class 1"])

bal_md = balance_df.assign(n=balance_df.n.map(f0), n_pos=balance_df.n_pos.map(f0), n_neg=balance_df.n_neg.map(f0),
                           pct_pos=balance_df.pct_pos.map(f2)).rename(columns={
    "set": "Set", "drawn_from": "Drawn from", "n": "n", "n_pos": "Positive", "n_neg": "Negative", "pct_pos": "% positive"})
ov = overlap_df.copy()
ov_md = pd.DataFrame({"Test rows": ["All", "Class 0", "Class 1"],
                      "n": [f0(ov.loc[k, "n_test"]) for k in ("all", "class0", "class1")],
                      "21-predictor profile in development": [f"{f0(ov.loc[k,'feature_profile_in_dev'])} ({ov.loc[k,'pct_feature_profile_in_dev']:.2f}%)" for k in ("all", "class0", "class1")],
                      "Full row (with label) in development": [f"{f0(ov.loc[k,'full_row_in_dev'])} ({ov.loc[k,'pct_full_row_in_dev']:.2f}%)" for k in ("all", "class0", "class1")],
                      "Novel profile (sensitivity subset)": [f0(ov.loc[k, "novel_feature_profile"]) for k in ("all", "class0", "class1")]})

report = f"""# Stage 1 Data Report: Verification and Partition

Generated by `{SCRIPT}` (config {CONFIG_VERSION}, master seed {MASTER_SEED}) on {env['run_utc']} (UTC), machine: {MACHINE}, Python {env['python']}.
Every number below was produced by this run. The raw CSV was read only; nothing was changed in it.

Dataset: `{os.path.basename(DATA_PATH)}`, SHA-256 `{DATA_SHA256}`.

## 1. Verification

All {len(checks_df)} checks passed.

{md_table(checks_df.rename(columns=str.capitalize))}

## 2. Variable coding

The coding below was checked against public sources, not assumed. All observed values match it (Section 1).

{md_table(pd.DataFrame(CODEBOOK, columns=["Variable", "Coding in this file", "Source"]))}

Sources: **UCI** = CDC Diabetes Health Indicators, UCI Machine Learning Repository, doi:10.24432/C53919 (variable table as mirrored by the AIM-AHEAD catalogue entry UCI_DS_891); **CDC-CB** = CDC, BRFSS 2015 LLCP codebook (codebook15_llcp.pdf); **CDC-CV** = CDC, BRFSS 2015 calculated variables, version 4 (2015_calculated_variables_version4.pdf).

Notes:
- The target is a self-reported past diagnosis ("ever told"). In the CDC source item (DIABETE3), "yes, only during pregnancy" is a separate code; how the derived file assigned it is not documented in the sources checked, so the thesis should not claim a specific mapping for it (D3 wording already avoids this).
- BMI is stored as a whole number. The CDC variable has two decimals, so the derived file rounded or truncated it; this does not affect the analysis but explains the integer values.
- The UCI description gives MentHlth and PhysHlth as 1–30 days; the file contains 0, which corresponds to the CDC code 88 ("None"). This is inferred from the codebook, not documented by UCI.

## 3. Class distribution

Positive (1, prediabetes or diabetes): {n_pos:,} ({100*n_pos/n_rows:.2f}%). Negative (0): {n_neg:,} ({100*n_neg/n_rows:.2f}%). Ratio negative/positive = {n_neg/n_pos:.2f}.

## 4. Repeated rows and profiles

| Quantity | Value |
| --- | --- |
| Exact repeated rows, all 22 columns (copies beyond the first) | {rep['full_repeated_rows']:,} ({100*rep['full_repeated_rows']/n_rows:.2f}%) |
| Distinct full rows | {rep['full_distinct_rows']:,} |
| Rows that belong to a repeated group (all copies) | {rep['full_rows_in_repeated_groups']:,} in {rep['full_repeated_groups']:,} groups; largest group {rep['full_largest_group']:,} rows |
| Repeated copies that are positive | {rep['full_repeated_rows_positive']:,} ({100*rep['full_repeated_rows_positive']/rep['full_repeated_rows']:.2f}% of copies) |
| Repeated 21-predictor profiles (label ignored) | {rep['feat_repeated_rows']:,} |
| Distinct 21-predictor profiles | {rep['feat_distinct_profiles']:,} |
| Profiles that occur with both labels | {rep['feat_conflicting_profiles']:,} profiles, {rep['feat_rows_in_conflicting_profiles']:,} rows |

Repeated rows are retained (D2). Only {100*rep['full_repeated_rows_positive']/rep['full_repeated_rows']:.2f}% of repeated copies are positive, against {100*n_pos/n_rows:.2f}% in the file, so repetition is concentrated in negative answer patterns. This is consistent with different respondents giving identical answers to 21 coarse items, but the data cannot confirm it. Profiles that occur with both labels show that the predictors do not determine the label for those respondents.

## 5. BMI and day counts

BMI: min {bmi_stats['min']:g}, 1st percentile {bmi_stats['p1']:g}, median {bmi_stats['p50']:g}, 99th percentile {bmi_stats['p99']:g}, max {bmi_stats['max']:g}; mean {bmi_stats['mean']:.2f} (SD {bmi_stats['sd']:.2f}). Rows with BMI ≥ 50: {bmi_stats['n_ge_50']:,}; ≥ 70: {bmi_stats['n_ge_70']:,}; ≥ 90: {bmi_stats['n_ge_90']:,}. All values lie inside the CDC valid range (12–99.99) and are retained without winsorization, as Chapter 3 §3.2.2 states.

MentHlth = 0 days: {days['MentHlth_pct_zero']:.2f}%; = 30 days: {days['MentHlth_pct_30']:.2f}%. PhysHlth = 0 days: {days['PhysHlth_pct_zero']:.2f}%; = 30 days: {days['PhysHlth_pct_30']:.2f}%. Both are concentrated at 0 with a secondary peak at the maximum of 30.

## 6. Descriptive statistics by class

Whole file, descriptive only (not used for any modelling choice). Binary: % with value 1. Ordinal and numerical: mean (SD); median [Q1–Q3].

{md_table(desc_md)}

## 7. Partition and index sets

Split: stratified 80:20, `train_test_split(stratify=target, random_state={SEEDS['partition']})`. Folds: `StratifiedKFold(5, shuffle=True, random_state={SEEDS['folds']})` on development rows sorted by row_id. Subsample: 50,000 development rows stratified by target × fold (seed {SEEDS['subsample_50k']}), each row keeping its development fold. SHAP sets: proportional stratification by target; the permutation background is drawn from the 500-row background and the permutation core from the 2,000-row global sample. Stability resamples: 5 bootstrap resamples within class for each of the 2,000 and 500 samples.

{md_table(bal_md)}

## 8. Test rows whose profile also occurs in development (D2)

{md_table(ov_md)}

The primary flag uses the 21 predictors only, since a model sees only those (S3); the full-row flag is reported for completeness. The novel-profile subset is the one used for the exploratory sensitivity analysis after test evaluation. Overlap is uneven by class ({ov.loc['class0','pct_feature_profile_in_dev']:.2f}% of negatives vs {ov.loc['class1','pct_feature_profile_in_dev']:.2f}% of positives), so the subset's positive share ({bd.loc['Test rows, novel 21-predictor profile','pct_pos']:.2f}%) is higher than the full test set's ({bd.loc['Test','pct_pos']:.2f}%). Average precision depends on prevalence, so a change in AP on this subset is not by itself evidence of optimism; prevalence-independent measures (ROC-AUC, sensitivity, specificity) are the cleaner comparison (open item O3).

## 9. Index files

Saved in `stage1_outputs/indices/`. Every later stage loads these files and never recomputes them (S1a).

{md_table(manifest_df[['file', 'rows', 'sha256', 'status']])}

## 10. Seeds

{md_table(pd.DataFrame(list(SEEDS.items()), columns=['Stream', 'Seed']))}

Seeds are derived as `numpy.random.SeedSequence({MASTER_SEED}).generate_state({len(SEED_STREAMS)})` in the listed order; the list is append-only.

## 11. Environment

Python {env['python']} on {env['platform']}; {env['cpu_count']} CPUs; {env['ram_gb']} GB RAM; GPU: {env['gpu']}. Package versions are in `requirements.txt` and `env/env_log_stage1_{MACHINE}.json`.
"""
with open(os.path.join(OUT_DIR, "stage1_data_report.md"), "w") as f:
    f.write(report)
print("Report written:", os.path.join(OUT_DIR, "stage1_data_report.md"))

UnicodeEncodeError: 'charmap' codec can't encode character '\u2265' in position 2851: character maps to <undefined>